In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)




In [5]:
batch_size = 2
seq_len = 4
d_model = 4

X = torch.randn(batch_size, seq_len, d_model)

X.shape

torch.Size([2, 4, 4])

In [7]:
X[0]

tensor([[-0.8371, -0.9224,  1.8113,  0.1606],
        [ 0.3672,  0.1754,  1.3852, -0.4459],
        [-1.2024,  0.7078, -1.0759,  0.5357],
        [ 1.1754,  0.5612, -0.4527, -0.7718]])

In [10]:
X[ 0, 0]

tensor([-0.8371, -0.9224,  1.8113,  0.1606])

In [12]:
X[1,-1] = 0

In [13]:
X[1,-1]

tensor([0., 0., 0., 0.])

In [14]:
d_model = 4
d_k = 4
d_v = 4


W_q = nn.Linear(d_model, d_k, bias=False)

W_k = nn.Linear(d_model, d_k, bias=False)

W_v = nn.Linear(d_model, d_v, bias=False)


Q = W_q(X)
K = W_k(X)
V = W_v(X)

print("X:", X.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

X: torch.Size([2, 4, 4])
Q: torch.Size([2, 4, 4])
K: torch.Size([2, 4, 4])
V: torch.Size([2, 4, 4])


In [15]:
W_q.weight

Parameter containing:
tensor([[-0.2523,  0.1524,  0.1057, -0.1275],
        [ 0.2980,  0.3399, -0.3626, -0.2669],
        [ 0.4578, -0.1687, -0.1773, -0.4838],
        [-0.2863,  0.1249, -0.0660, -0.3629]], requires_grad=True)

In [16]:
W_q.weight.shape

torch.Size([4, 4])

In [17]:
print("W_q:", W_q.weight.shape)
print("W_k:", W_k.weight.shape)
print("W_v:", W_v.weight.shape)

W_q: torch.Size([4, 4])
W_k: torch.Size([4, 4])
W_v: torch.Size([4, 4])


Q (Query): Token'ın hangi bilgiyi aradığını temsil eder.
K (Key): Diğer token'larla eşleşme hesaplamasında kullanılır.
Q × Kᵀ: Her Query token'ının bütün Key token'larıyla ilişki skorlarını hesaplar.

In [19]:
K_T = K.transpose(-2, -1)

scores = torch.matmul(Q, K_T)

print("Attention Scores Shape:", scores.shape)
print("\nİlk metnin skor matrisi:")
print(scores[0])

Attention Scores Shape: torch.Size([2, 4, 4])

İlk metnin skor matrisi:
tensor([[-1.9122, -0.4077,  0.4232,  1.1752],
        [-0.4417, -0.2057,  0.0242,  0.2627],
        [ 0.2943,  0.1704,  0.0702, -0.1678],
        [ 1.2299,  0.1053, -0.4182, -0.7566]], grad_fn=<SelectBackward0>)


In [20]:
scaled_scores = scores / math.sqrt(d_k)

print("Original Scores:")
print(scores[0])

print("\nScaled Scores:")
print(scaled_scores[0])

Original Scores:
tensor([[-1.9122, -0.4077,  0.4232,  1.1752],
        [-0.4417, -0.2057,  0.0242,  0.2627],
        [ 0.2943,  0.1704,  0.0702, -0.1678],
        [ 1.2299,  0.1053, -0.4182, -0.7566]], grad_fn=<SelectBackward0>)

Scaled Scores:
tensor([[-0.9561, -0.2038,  0.2116,  0.5876],
        [-0.2209, -0.1029,  0.0121,  0.1314],
        [ 0.1472,  0.0852,  0.0351, -0.0839],
        [ 0.6150,  0.0526, -0.2091, -0.3783]], grad_fn=<SelectBackward0>)


In [21]:
attention_weights = F.softmax(
    scaled_scores,
    dim=-1
)

print("Attention Weights Shape:", attention_weights.shape)

print("\nİlk metnin Attention matrisi:")
print(attention_weights[0])

print("\nSatır toplamları:")
print(attention_weights[0].sum(dim=-1))

Attention Weights Shape: torch.Size([2, 4, 4])

İlk metnin Attention matrisi:
tensor([[0.0908, 0.1926, 0.2918, 0.4249],
        [0.2079, 0.2339, 0.2624, 0.2957],
        [0.2757, 0.2591, 0.2464, 0.2188],
        [0.4204, 0.2396, 0.1844, 0.1557]], grad_fn=<SelectBackward0>)

Satır toplamları:
tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [23]:
attention_output = torch.matmul(
    attention_weights,
    V
)

print("Attention Output Shape:", attention_output.shape)

print("\nİlk metnin Attention çıktısı:")
print(attention_output[0])

Attention Output Shape: torch.Size([2, 4, 4])

İlk metnin Attention çıktısı:
tensor([[ 0.0116, -0.0557, -0.0493,  0.1329],
        [ 0.0064, -0.0255,  0.0542,  0.0846],
        [ 0.0028, -0.0077,  0.1136,  0.0573],
        [ 0.0154,  0.0240,  0.2310, -0.0179]], grad_fn=<SelectBackward0>)


In [25]:
valid_mask = torch.tensor([
    [True, True, True, True],
    [True, True, True, False]
])

print(valid_mask.shape)
print(valid_mask.dtype)

torch.Size([2, 4])
torch.bool


In [26]:
key_mask = valid_mask.unsqueeze(1)

print("Original Mask:", valid_mask.shape)
print("Key Mask:", key_mask.shape)

Original Mask: torch.Size([2, 4])
Key Mask: torch.Size([2, 1, 4])


In [31]:
masked_scores = scaled_scores.masked_fill(
    ~key_mask,
    float("-inf")
)

print("Masked Scores Shape:", masked_scores.shape)

print("\nİkinci metnin maskelenmiş skorları:")
print(masked_scores[1])

Masked Scores Shape: torch.Size([2, 4, 4])

İkinci metnin maskelenmiş skorları:
tensor([[-0.0158,  0.0525, -0.0978,    -inf],
        [ 0.1148, -0.2558,  0.6047,    -inf],
        [-0.0588,  0.4593, -0.3849,    -inf],
        [ 0.0000,  0.0000,  0.0000,    -inf]], grad_fn=<SelectBackward0>)


In [32]:
masked_weights = F.softmax(
    masked_scores,
    dim=-1
)

print("Masked Attention Shape:", masked_weights.shape)

print("\nİkinci metnin Attention ağırlıkları:")
print(masked_weights[1])

print("\nPadding konumunun ağırlıkları:")
print(masked_weights[1, :, 3])

Masked Attention Shape: torch.Size([2, 4, 4])

İkinci metnin Attention ağırlıkları:
tensor([[0.3342, 0.3579, 0.3079, 0.0000],
        [0.3010, 0.2078, 0.4912, 0.0000],
        [0.2941, 0.4937, 0.2122, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000]], grad_fn=<SelectBackward0>)

Padding konumunun ağırlıkları:
tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)


In [33]:
print("Padding Weights:", masked_weights[1, :, 3])

print("Row Sums:", masked_weights[1].sum(dim=-1))

Padding Weights: tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)
Row Sums: tensor([1., 1., 1., 1.], grad_fn=<SumBackward1>)


In [34]:
attention_without_mask = torch.matmul(
    attention_weights,
    V
)

attention_with_mask = torch.matmul(
    masked_weights,
    V
)

print("Without Mask:", attention_without_mask.shape)
print("With Mask:", attention_with_mask.shape)

Without Mask: torch.Size([2, 4, 4])
With Mask: torch.Size([2, 4, 4])


In [36]:
print("Maskesiz Attention Ağırlıkları:")
print(attention_weights[1, 0])

print("\nMaskeli Attention Ağırlıkları:")
print(masked_weights[1, 0])

Maskesiz Attention Ağırlıkları:
tensor([0.2495, 0.2672, 0.2299, 0.2535], grad_fn=<SelectBackward0>)

Maskeli Attention Ağırlıkları:
tensor([0.3342, 0.3579, 0.3079, 0.0000], grad_fn=<SelectBackward0>)


## GÖREV 6 — Attention Mekanizmasını Yorumlama

### Soru 1 — Query, Key ve Value

Aynı giriş Tensor'ından Q, K ve V olmak üzere üç farklı projeksiyon oluşturuyoruz.

Query (Q), token'ın hangi bilgileri aradığını temsil eder. Key (K), token'ların Query ile karşılaştırılacak özelliklerini temsil eder.

Query ve Key arasında Dot Product hesaplayarak token'lar arasındaki ilişki skorlarını elde ederiz.

Bu skorları ölçeklendirip Softmax uyguladıktan sonra Attention ağırlıklarını oluştururuz.

Value (V) ise aktarılacak bilgiyi temsil eder. Attention ağırlıklarıyla Value vektörlerini birleştirerek yeni token temsillerini elde ederiz.

Bu nedenle Q, K ve V farklı öğrenilebilir ağırlık matrisleriyle oluşturulur.

### Soru 2 — Scaling

Query ve Key vektörlerinin boyutları büyüdükçe Dot Product skorları çok büyük değerlere ulaşabilir.

Bu skorları doğrudan Softmax'a vermek, Attention ağırlıklarının aşırı keskin dağılmasına neden olabilir. Bu durum gradyanları küçülterek öğrenmeyi zorlaştırabilir.

Bu yüzden skorları Key boyutunun kareköküne bölüyoruz.

**Scaling, Softmax'ın daha kararlı çalışmasına yardımcı olur.**

### Soru 3 — Padding Mask

Padding Mask, gerçek bilgi taşımayan PAD token'larının Attention hesabına katkısını engellemek için kullanılır.

Maskeyi Softmax'tan sonra uygularsak Softmax ağırlıkları PAD token'ını da dikkate alarak dağıtır.

Daha sonra PAD ağırlığını sıfırlasak bile diğer token'ların ağırlıkları yeniden hesaplanmaz. Bu nedenle ağırlıkların toplamı 1'den küçük kalabilir.

Bu sorunu önlemek için Padding Mask'i Softmax'tan önce uyguluyoruz.

PAD skorlarını `-inf` yaparak Softmax sonucunda bu konumların sıfır Attention ağırlığı almasını sağlıyoruz.

### Soru 4 — Self-Attention ve RNN Farkı

RNN modelleri dizileri sırayla işler. Her adımda bir önceki hidden state üzerinden geçmiş bilgileri yeni adıma taşır.

Self-Attention ise her token'ın aynı dizideki diğer token'larla olan ilişkilerini Query ve Key vektörleri üzerinden hesaplar.

Elde edilen Attention ağırlıklarıyla Value vektörlerini birleştirerek her token'ın temsilini günceller.

RNN'de uzak token'lardan gelen bilgiler ara hidden state'lerden geçerken Self-Attention'da izin verilen token'lar arasında doğrudan ilişki hesaplanabilir.

Ayrıca Self-Attention, token'lar arasındaki ilişkileri matris işlemleriyle paralel olarak hesaplayabilir.

Bu özellikler, özellikle uzun dizilerdeki ilişkilerin modellenmesi açısından avantaj sağlayabilir.

In [38]:
assert Q.shape == (2, 4, 4)
assert K.shape == (2, 4, 4)
assert V.shape == (2, 4, 4)

assert attention_without_mask.shape == (2, 4, 4)
assert attention_with_mask.shape == (2, 4, 4)

print("Test 1 — Q/K/V Shapes: PASSED")


assert torch.allclose(
    attention_weights.sum(dim=-1),
    torch.ones_like(attention_weights.sum(dim=-1)),
    atol=1e-6
)

assert torch.allclose(
    masked_weights.sum(dim=-1),
    torch.ones_like(masked_weights.sum(dim=-1)),
    atol=1e-6
)

print("Test 2 — Attention Weights Sum: PASSED")


assert torch.allclose(
    masked_weights[1, :, 3],
    torch.zeros_like(masked_weights[1, :, 3]),
    atol=1e-6
)

print("Test 3 — Padding Mask: PASSED")

Test 1 — Q/K/V Shapes: PASSED
Test 2 — Attention Weights Sum: PASSED
Test 3 — Padding Mask: PASSED
